# MAPBIOMAS-CARBONO · estatísticas — análise de escritório

Notebook montado a partir de `cells_for_collab_notebook/*.py` (cada `.py` é uma célula).
Rode **na ordem**: a biblioteca (`00`–`06`) primeiro; depois as etapas (`10`–`16`).
As estatísticas são computadas no GEE (`toolkits/statistics/v00/run/`) e apenas lidas aqui
(exceção: `16_maps`, opcional).


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
00_config.py · CONFIGURAÇÃO + paths/artefatos.

Primeira célula do notebook (rode ANTES de todas as outras). Não faz import de
nada próprio: cole o conteúdo como a PRIMEIRA célula (ou use %run) para deixar
`CONF`, os paths e helpers no namespace do runtime.

Pipeline (ver README.md): 00_config → 01_legends → 02_schema → 03_loaders →
04_qa → 05_aggregations → 06_references → 10_auth_drive → 11_consolidate →
12_join_largo → 13_figures → 14_report → 15_references → 16_maps.
"""
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

# ---------------------------------------------------------------------------
# CONFIG (ajustável a cada execução — a 10_auth_drive preenche via DRIVE_ROOT)
# ---------------------------------------------------------------------------
CONF = {
    "DRIVE_ROOT": "/content/drive/MyDrive/mapbiomas_carbono/pantanal",
    "CSV_DIR": "estatisticas",  # CSVs exportados pelo GEE (run/*)
    "FIG_DIR": "figures",       # biblioteca de figuras (espelha o repo figures/)
    "OUT_DIR": "relatorio",     # XLSX + .md + consolidados
    "YEAR": 2023,
    # --- escopo esperado dos arquivos ---
    "FOREST_VERSIONS": ["v03", "v04", "v05", "v06", "v07", "v08"],
    "GRASSLAND_PRODUCTS": ["biomass", "anpp", "carbon"],
    "GRASSLAND_REF_VERSION": "v02",
    "MISSING_TOL": 1.0,     # t (QA de somas)
    "DENSITY_MAX": 400.0,   # t/ha (QA de densidade implícita)
    # --- filtro de estrato def_sec (None/[] = sem filtro) ---
    # rótulos de DEF_SEC_VEG_LEGEND; ex.: ["Vegetação Primária"] (código 2)
    "DEF_SEC_FILTER": ["Vegetação Primária"],
    # --- dados de REFERÊNCIA (15_references) ---
    "CARBON_FRACTION": 0.47,  # AGB (t) -> C (t), IPCC default
    "REF_MIN_VALID": 0.05,    # mín. fração de área válida p/ comparar ref
    "REF_TERRITORIO": "Pantanal (bioma)",  # território filtrado nos CSVs
    # --- figuras ---
    "FIGURE_DPI": 120,
    # --- mapas OPCIONAIS (off por padrão — recomputam no Earth Engine) ---
    "FAZER_MAPAS": False,
    "GRID_M": 10000,    # grade de agregação do mapa de médias (10 km)
    "MAP_FOREST_VERSION": "v07",  # versão do forest usada no mapa (16_maps)
    "C_RED": "#8c2d04", "C_FOREST": "#1a9850", "C_GRASS": "#fdae61",
    "C_OTHER": "#969696",
}
_MANIFEST: list[str] = []


# ---------------------------------------------------------------------------
# Paths / figuras / artefatos
# ---------------------------------------------------------------------------
def csv_dir() -> Path:
    d = Path(CONF["DRIVE_ROOT"]) / CONF["CSV_DIR"]
    d.mkdir(parents=True, exist_ok=True)
    return d


def fig_dir() -> Path:
    d = Path(CONF["DRIVE_ROOT"]) / CONF["FIG_DIR"]
    d.mkdir(parents=True, exist_ok=True)
    return d


def out_dir() -> Path:
    d = Path(CONF["DRIVE_ROOT"]) / CONF["OUT_DIR"]
    d.mkdir(parents=True, exist_ok=True)
    return d


def save_fig(fig, name: str) -> str:
    path = fig_dir() / name
    fig.savefig(path, dpi=CONF["FIGURE_DPI"], bbox_inches="tight")
    plt.close(fig)
    _MANIFEST.append(str(path))
    print(f"[FIG] {path}")
    return str(path)


def write_artifact(path: Path, content) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    if isinstance(content, str):
        path.write_text(content, encoding="utf-8")
    else:
        path.write_text(json.dumps(content, ensure_ascii=False, indent=2),
                        encoding="utf-8")
    _MANIFEST.append(str(path))
    print(f"[OUT] {path}")


def manifest() -> list[str]:
    return list(_MANIFEST)


print("=" * 68)
print("statistics · 00_config · CONF + paths carregados")
print("     DRIVE_ROOT:", CONF["DRIVE_ROOT"], "· CSV_DIR:", CONF["CSV_DIR"],
      "· OUT_DIR:", CONF["OUT_DIR"])
print("=" * 68)


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
01_legends.py · LEGENDAS (LULC Coleção 11 + def_sec) e helpers de rótulo.

Espelho do módulo GEE `engine/module_legends.js`. Rode após 00_config.
"""
# ---------------------------------------------------------------------------
# Assets oficiais (MapBiomas C11)
# ---------------------------------------------------------------------------
C11_ASSET = (
    "projects/mapbiomas-public/assets/brazil/lulc/collection11/"
    "mapbiomas_brazil_collection11_coverage_v3"
)
DEF_SEC_ASSET = (
    "projects/mapbiomas-public/assets/brazil/lulc/collection11/"
    "mapbiomas_brazil_collection11_deforestation_secondary_vegetation_v5"
)


def defsec_band(year: int) -> str:
    return f"classification_{year}"


# ---- Legenda def_sec (0..7) --------------------------------------------------
DEF_SEC_VEG_LEGEND = {
    0: "Não Observado",
    1: "Antrópico",
    2: "Vegetação Primária",
    3: "Vegetação Secundária",
    4: "Desmatamento Primário",
    5: "Recrescimento",
    6: "Desmatamento Secundário",
    7: "Outras Transições",
}
DEFSEC_MODS = [0, 1, 2, 3, 4, 5, 6, 7]
DEFSEC_TIPOS = {
    0: "nao_observado",
    1: "antropico",
    2: "primaria",
    3: "secundaria",
    4: "desmatamento",
    5: "recrescimento",
    6: "desmatamento",
    7: "transicao",
}


def defsec_label(mod) -> str:
    return DEF_SEC_VEG_LEGEND.get(int(mod), f"Mod {int(mod)}")


def defsec_tipo(mod) -> str:
    return DEFSEC_TIPOS.get(int(mod), "outro")


DEF_SEC_CODE = {v: k for k, v in DEF_SEC_VEG_LEGEND.items()}


def defsec_code(label):
    """Rótulo da legenda def_sec (por extenso) → código 0..7 (-1 se fora)."""
    if pd.isna(label):
        return -1
    if isinstance(label, (int, float, np.integer, np.floating)):
        return int(label)
    return DEF_SEC_CODE.get(str(label).strip(), -1)


# ---- Legendas oficiais por nível (código → nome) -----------------------------
LULC_NIVEIS = {
    0: "Não observado", 1: "Nível 1", 3: "Nível 2", 4: "Nível 2",
    5: "Nível 2", 6: "Nível 2", 7: "Nível 2", 49: "Nível 2", 10: "Nível 1",
    11: "Nível 2", 12: "Nível 2", 77: "Nível 2", 32: "Nível 2",
    29: "Nível 2", 50: "Nível 2", 13: "Nível 2", 84: "Nível 2",
    14: "Nível 1", 15: "Nível 2", 18: "Nível 2", 19: "Nível 3",
    39: "Nível 4", 20: "Nível 4", 40: "Nível 4", 62: "Nível 4",
    41: "Nível 4", 36: "Nível 3", 46: "Nível 4", 47: "Nível 4",
    35: "Nível 4", 48: "Nível 4", 9: "Nível 2", 21: "Nível 2",
    22: "Nível 1", 23: "Nível 2", 24: "Nível 2", 30: "Nível 2",
    75: "Nível 2", 91: "Nível 2", 25: "Nível 2", 26: "Nível 1",
    33: "Nível 2", 31: "Nível 3", 27: "Não observado",
}

LULC_NIVEL_0 = {
    0: "Não observado", 1: "Natural", 3: "Natural", 4: "Natural",
    5: "Natural", 6: "Natural", 7: "Natural", 49: "Natural", 10: "Natural",
    11: "Natural", 12: "Natural", 77: "Natural", 32: "Natural",
    29: "Antrópico", 50: "Natural", 13: "Natural", 84: "Natural",
    14: "Antrópico", 15: "Antrópico", 18: "Antrópico", 19: "Antrópico",
    39: "Antrópico", 20: "Antrópico", 40: "Antrópico", 62: "Antrópico",
    41: "Antrópico", 36: "Antrópico", 46: "Antrópico", 47: "Antrópico",
    35: "Antrópico", 48: "Antrópico", 9: "Antrópico", 21: "Antrópico",
    22: "Antrópico", 23: "Antrópico", 24: "Antrópico", 30: "Antrópico",
    75: "Antrópico", 91: "Antrópico", 25: "Antrópico", 26: "Antrópico",
    33: "Antrópico", 31: "Antrópico", 27: "Não observado",
}

LULC_NIVEL_1 = {
    0: "Não observado", 1: "Floresta", 3: "Floresta", 4: "Floresta",
    5: "Floresta", 6: "Floresta", 7: "Floresta", 49: "Floresta",
    10: "Vegetação Herbácea e Arbustiva", 11: "Vegetação Herbácea e Arbustiva",
    12: "Vegetação Herbácea e Arbustiva", 77: "Vegetação Herbácea e Arbustiva",
    32: "Vegetação Herbácea e Arbustiva", 29: "Área não vegetada",
    50: "Vegetação Herbácea e Arbustiva",
    13: "Vegetação Herbácea e Arbustiva", 84: "Vegetação Herbácea e Arbustiva",
    14: "Agropecuária", 15: "Agropecuária", 18: "Agropecuária",
    19: "Agropecuária", 39: "Agropecuária", 20: "Agropecuária",
    40: "Agropecuária", 62: "Agropecuária", 41: "Agropecuária",
    36: "Agropecuária", 46: "Agropecuária", 47: "Agropecuária",
    35: "Agropecuária", 48: "Agropecuária", 9: "Agropecuária",
    21: "Agropecuária", 22: "Área não vegetada", 23: "Área não vegetada",
    24: "Área não vegetada", 30: "Área não vegetada",
    75: "Área não vegetada", 91: "Área não vegetada",
    25: "Área não vegetada", 26: "Corpos D´água", 33: "Corpos D´água",
    31: "Corpos D´água", 27: "Não observado",
}

LULC_NIVEL_1_1 = {
    0: "Não observado", 1: "Formação Florestal", 3: "Formação Florestal",
    4: "Formação Savânica", 5: "Formação Florestal",
    6: "Formação Florestal", 7: "Formação Florestal", 49: "Formação Savânica",
    10: "Formação Campestre", 11: "Campo Alagado e Área Pantanosa",
    12: "Formação Campestre", 77: "Formação Campestre", 32: "Formação Campestre",
    29: "Outros", 50: "Formação Campestre", 13: "Formação Campestre",
    84: "Formação Campestre", 14: "Agropecuária", 15: "Pastagem",
    18: "Agricultura", 19: "Agricultura", 39: "Agricultura",
    20: "Agricultura", 40: "Agricultura", 62: "Agricultura",
    41: "Agricultura", 36: "Agricultura", 46: "Agricultura",
    47: "Agricultura", 35: "Agricultura", 48: "Agricultura",
    9: "Silvicultura", 21: "Mosaico de Usos", 22: "Outros", 23: "Outros",
    24: "Outros", 30: "Outros", 75: "Outros", 91: "Outros", 25: "Outros",
    26: "Outros", 33: "Outros", 31: "Outros", 27: "Não observado",
}

LULC_NIVEL_2 = {
    0: "Não observado", 1: "Floresta", 3: "Formação Florestal",
    4: "Formação Savânica", 5: "Mangue", 6: "Floresta Alagável",
    7: "Savana Alagada", 49: "Restinga Arbórea",
    10: "Vegetação Herbácea e Arbustiva", 11: "Campo Alagado e Área Pantanosa",
    12: "Formação Campestre", 77: "Formação Herbáceo Arbustiva",
    32: "Apicum", 29: "Afloramento Rochoso", 50: "Restinga herbácea",
    13: "Outras Formações não Florestais", 84: "Marisma",
    14: "Agropecuária", 15: "Pastagem", 18: "Agricultura",
    19: "Agricultura", 39: "Agricultura", 20: "Agricultura",
    40: "Agricultura", 62: "Agricultura", 41: "Agricultura",
    36: "Agricultura", 46: "Agricultura", 47: "Agricultura",
    35: "Agricultura", 48: "Agricultura", 9: "Silvicultura",
    21: "Mosaico de Usos", 22: "Área não vegetada", 23: "Praia, Duna e Areal",
    24: "Área Urbanizada", 30: "Mineração", 75: "Usina Fotovoltaica",
    91: "Energia Eólica", 25: "Outras Áreas não Vegetadas",
    26: "Corpos D´água", 33: "Rio, Lago e Oceano", 31: "Aquicultura",
    27: "Não observado",
}

LULC_NIVEL_3 = {
    0: "Não observado", 1: "Floresta", 3: "Formação Florestal",
    4: "Formação Savânica", 5: "Mangue", 6: "Floresta Alagável",
    7: "Savana Alagada", 49: "Restinga Arbórea",
    10: "Formação Natural não Florestal", 11: "Campo Alagado e Área Pantanosa",
    12: "Formação Campestre", 77: "Formação Herbáceo Arbustiva",
    32: "Apicum", 29: "Afloramento Rochoso", 50: "Restinga herbácea",
    13: "Outras Formações não Florestais", 84: "Marisma",
    14: "Agropecuária", 15: "Pastagem", 18: "Agricultura",
    19: "Lavoura Temporária", 39: "Lavoura Temporária", 20: "Lavoura Temporária",
    40: "Lavoura Temporária", 62: "Lavoura Temporária",
    41: "Lavoura Temporária", 36: "Lavoura Perene", 46: "Lavoura Perene",
    47: "Lavoura Perene", 35: "Lavoura Perene", 48: "Lavoura Perene",
    9: "Silvicultura", 21: "Mosaico de Usos", 22: "Área não vegetada",
    23: "Praia, Duna e Areal", 24: "Área Urbanizada", 30: "Mineração",
    75: "Usina Fotovoltaica", 91: "Energia Eólica",
    25: "Outras Áreas não Vegetadas", 26: "Corpos D´água",
    33: "Rio, Lago e Oceano", 31: "Aquicultura", 27: "Não observado",
}

LULC_NIVEL_4 = {
    0: "Não observado", 1: "Floresta", 3: "Formação Florestal",
    4: "Formação Savânica", 5: "Mangue", 6: "Floresta Alagável",
    7: "Savana Alagada", 49: "Restinga Arbórea",
    10: "Formação Natural não Florestal", 11: "Campo Alagado e Área Pantanosa",
    12: "Formação Campestre", 77: "Formação Herbáceo Arbustiva",
    32: "Apicum", 29: "Afloramento Rochoso", 50: "Restinga herbácea",
    13: "Outras Formações não Florestais", 84: "Marisma",
    14: "Agropecuária", 15: "Pastagem", 18: "Agricultura",
    19: "Lavoura Temporária", 39: "Soja", 20: "Cana", 40: "Arroz",
    62: "Algodão", 41: "Outras Lavouras Temporárias", 36: "Lavoura Perene",
    46: "Café", 47: "Citrus", 35: "Dendê", 48: "Outras Lavouras Perenes",
    9: "Silvicultura", 21: "Mosaico de Usos", 22: "Área não Vegetada",
    23: "Praia, Duna e Areal", 24: "Área Urbanizada", 30: "Mineração",
    75: "Usina Fotovoltaica", 91: "Energia Eólica",
    25: "Outras Áreas não Vegetadas", 26: "Corpos D´água",
    33: "Rio, Lago e Oceano", 31: "Aquicultura", 27: "Não observado",
}

# Códigos de lavoura com nome específico no nível 4 (n0 usará nível 4).
LAVOURA_N4_CODES = [19, 20, 35, 36, 39, 40, 41, 46, 47, 48, 62]


def _clamp(d: dict, code) -> str:
    try:
        return d[int(code)]
    except (KeyError, TypeError, ValueError):
        return f"Classe {int(code)}"


def n0_label(code) -> str:
    c = int(code)
    if c in LAVOURA_N4_CODES and c in LULC_NIVEL_4:
        return LULC_NIVEL_4[c]
    return _clamp(LULC_NIVEL_2, c)


def n1_of(code) -> str:
    return _clamp(LULC_NIVEL_1, code)


def n2_of(code) -> str:
    return _clamp(LULC_NIVEL_2, code)


def natureza_of(code) -> str:
    return _clamp(LULC_NIVEL_0, code)


def marcador_from_n1(n1: str) -> str:
    if n1 == "Floresta":
        return "florestal"
    if n1 == "Vegetação Herbácea e Arbustiva":
        return "campestre"
    return "outros"


def marcador_of(code) -> str:
    return marcador_from_n1(n1_of(code))


def info(code):
    n1 = n1_of(code)
    return {
        "n0": n0_label(code),
        "n1": n1,
        "n2": n2_of(code),
        "natureza": natureza_of(code),
        "marcador": marcador_from_n1(n1),
    }

print("[OK] 01_legends · legendas LULC C11 + def_sec carregadas")


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
02_schema.py · Schema "gráfico-ready" dos CSVs do GEE → nomes internos.

Colunas de PRODUTO (forest/grassland):
  'cobertura (n0)' → 'n0' · 'Área (ha)' → 'area_ha'
  'Biomassa acima do solo (t MS)' → 'biomassa_t'
  'Carbono acima do solo (t C)' → 'estoque_t'
  'Produtividade primária líquida acima do solo (t MS/ano)' → 'anpp_t'
Colunas de MÉTRICA das REFERÊNCIAS: ver REF_COL_MAP.

Rode após 01_legends.
"""
# ---------------------------------------------------------------------------
# Schema "gráfico-ready" dos CSVs → nomes internos canônicos.
# ---------------------------------------------------------------------------
GRAPH_COLS = {
    "cobertura (n0)": "n0",
    "cobertura (n1)": "n1",
    "cobertura (n2)": "n2",
    "cobertura (n3)": "n3",
    "cobertura (n4)": "n4",
    "Desmatamento e vegetação secundária": "def_sec",
    "Área (ha)": "area_ha",
    "Biomassa acima do solo (t MS)": "biomassa_t",
    "Carbono acima do solo (t C)": "estoque_t",
    "Produtividade primária líquida acima do solo (t MS/ano)": "anpp_t",
    "período": "periodo",
}

# Colunas de MÉTRICA das REFERÊNCIAS (nome descritivo → stem interno).
# O `*` marca a métrica derivada NO ESTÁGIO DE ESTATÍSTICA (ex.: AGB→C).
REF_COL_MAP = {
    "Biomassa acima do solo (t MS)": "agb_tB",
    "Carbono acima do solo (t C)*": "agb_tC",
    "Carbono acima do solo (t C)": "agc_tC",
    "Biomassa abaixo do solo (t MS)": "bgb_tB",
    "Carbono abaixo do solo (t C)*": "bgb_tC",
    "Carbono abaixo do solo (t C)": "bgc_tC",
    "Carbono na madeira morta (t C)": "cdw_tC",
    "Carbono na serrapilheira (t C)": "litter_tC",
    "Carbono vivo (t C)": "live_tC",
    "Carbono total (t C)": "total_tC",
    "Incerteza (t C)": "incerteza_tC",
}


def _drop_ee_columns(df: pd.DataFrame) -> pd.DataFrame:
    return df.drop(
        columns=[c for c in ("system:index", ".geo") if c in df.columns] or [],
        errors="ignore")


def _derive_territorio(df: pd.DataFrame) -> pd.DataFrame:
    """Compatibilidade: o CSV não traz `territorio`; deriva de `Bioma`."""
    if "territorio" not in df.columns and "Bioma" in df.columns:
        df["territorio"] = df["Bioma"].astype(str) + " (bioma)"
    return df


def _filter_defsec(df: pd.DataFrame) -> pd.DataFrame:
    """Mantém apenas as linhas cujo `def_sec` esteja em CONF['DEF_SEC_FILTER'].

    Sem filtro (None/[]), devolve o df como está. Ex.: `["Vegetação Primária"]`.
    """
    labels = CONF.get("DEF_SEC_FILTER") or []
    if not labels or "def_sec" not in df.columns:
        return df
    return df[df["def_sec"].astype(str).isin([str(x) for x in labels])].copy()


def _remap_products(df: pd.DataFrame) -> pd.DataFrame:
    """Renomeia colunas de PRODUTO (forest/grassland) para nomes internos."""
    df = df.rename(columns={k: v for k, v in GRAPH_COLS.items()
                            if k in df.columns})
    return _derive_territorio(df)


def _remap_refs(df: pd.DataFrame) -> pd.DataFrame:
    """Renomeia as colunas de MÉTRICA das REFERÊNCIAS para nomes internos."""
    df = df.rename(columns={k: v for k, v in REF_COL_MAP.items()
                            if k in df.columns})
    return _derive_territorio(df)


def _read(pattern: str) -> pd.DataFrame:
    files = sorted(csv_dir().glob(pattern))
    if not files:
        return pd.DataFrame()
    frames = [_drop_ee_columns(pd.read_csv(f)) for f in files]
    return pd.concat(frames, ignore_index=True)


def _clean_schema(df: pd.DataFrame) -> pd.DataFrame:
    """Padroniza o schema do consolidado (produtos forest/grassland).

    O GEE exporta apenas COMBINAÇÕES (classe × def_sec; `período` nas séries
    campestres): as colunas `natureza`/`marcador`/`def_sec_code` NÃO vêm do
    CSV e são derivadas em memória a partir da classe LULC (01_legends).
    """
    if df is None or not len(df):
        return pd.DataFrame()
    if "nivel" in df.columns:
        df = df[df["nivel"] == "n0"].copy()
    df = df.drop(columns=[c for c in ("nivel", "grupo") if c in df.columns],
                 errors="ignore")
    df = _remap_products(df)
    if "classe" in df.columns:
        df["classe"] = pd.to_numeric(df["classe"], errors="coerce")
    # sentinela de nodata dos produtos (ex.: grassland ~ ±9.22e17) → NaN
    for c in ("area_ha", "estoque_t", "biomassa_t", "anpp_t", "incerteza_t"):
        if c in df.columns:
            df.loc[df[c].abs() >= 1e15, c] = np.nan
    df = _filter_defsec(df)
    if "def_sec" in df.columns:
        df["def_sec_code"] = df["def_sec"].map(defsec_code)
    if "natureza" not in df.columns and "classe" in df.columns:
        df["natureza"] = df["classe"].map(lambda c: natureza_of(c)
                                          if pd.notna(c) else None)
    if "marcador" not in df.columns and "classe" in df.columns:
        df["marcador"] = df["classe"].map(lambda c: marcador_of(c)
                                          if pd.notna(c) else None)
    return df

print("[OK] 02_schema · GRAPH_COLS + REF_COL_MAP + remap carregados")


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
03_loaders.py · Leitura canônica dos CSVs do GEE (forest/grassland).

Rode após 02_schema. Deixa as tabelas LONGAS por versão/produto no namespace.
"""
import re

# ---------------------------------------------------------------------------
# Forest — CSV LARGO (todas as versões em colunas) → longo por versão
# ---------------------------------------------------------------------------
def load_forest() -> pd.DataFrame:
    """Carrega o CSV LARGO florestal (stats_forest_*2023*.csv) e o desmembra em
    formato longo por versão, a partir das colunas '<vXX> Biomassa acima do solo
    (t MS)' / '<vXX> Carbono acima do solo (t C)*' e das incertezas. A biomassa
    é a nativa (`agbd`); o carbono é derivado no stats (mapeado p/ `estoque_t`)."""
    df = _read("stats_forest_*2023*.csv")
    if not len(df):
        return pd.DataFrame()
    bio_re = re.compile(r"^(v\d+)\s+Biomassa acima do solo \(t MS\)$")
    bio_cols = [c for c in df.columns if bio_re.match(str(c))]
    if not bio_cols:
        return _clean_schema(df)  # schema antigo
    frames = []
    for c in bio_cols:
        v = bio_re.match(str(c)).group(1)
        part = df.copy()
        part["version"] = v
        part["biomassa_t"] = part[c]
        carb = f"{v} Carbono acima do solo (t C)*"
        incb = f"{v} Incerteza (t MS)"
        incc = f"{v} Incerteza (t C)*"
        part["estoque_t"] = part[carb] if carb in df.columns else pd.NA
        part["incerteza_t"] = part[incc] if incc in df.columns else pd.NA
        part["incerteza_tb"] = part[incb] if incb in df.columns else pd.NA
        frames.append(part)
    long = pd.concat(frames, ignore_index=True)
    drop_re = re.compile(
        r"^v\d+\s+(Carbono acima do solo \(t C\)\*|Incerteza \(t (MS|C)\)\*?)$")
    drop = bio_cols + [c for c in df.columns if drop_re.match(str(c))]
    long = long.drop(columns=drop, errors="ignore")
    return _clean_schema(long)


# ---------------------------------------------------------------------------
# Grassland — principal ANUAL + biomassa/ANPP mensais LONGOS
# ---------------------------------------------------------------------------
def load_grassland() -> pd.DataFrame:
    """Concatena stats_grassland_*_2023*.csv (principal ANUAL + biomassa/ANPP
    mensais LONGOS — 1 linha por combinação classe × def_sec × mês)."""
    return _clean_schema(_read("stats_grassland_*2023*.csv"))


# ---------------------------------------------------------------------------
# Presença de arquivos
# ---------------------------------------------------------------------------
_gv = CONF["GRASSLAND_REF_VERSION"]
EXPECTED = {
    "forest": ["stats_forest_"],
    "grassland": [f"stats_grassland_{_gv}_",
                  f"stats_grassland_{_gv}_biomassa_mensal_",
                  f"stats_grassland_{_gv}_anpp_mensal_"],
    "references": ["stats_references_"],
}


def check_presence(detect: str | None = None) -> list[str]:
    """Retorna lista de arquivos ausentes (detect via glob simples)."""
    missing = []
    for grp, pats in EXPECTED.items():
        for p in pats:
            if not list(csv_dir().glob(f"{p}*.csv")):
                missing.append((grp, p))
    for grp, m in missing:
        js = "run_references.js" if grp == "references" else "run_pantanal.js"
        print(f"[AUSENTE] {m}*.csv → rode o {js} no GEE e re-exporte"
              f" para {csv_dir()}")
    if not missing:
        print("[OK] todos os arquivos esperados presentes em", csv_dir())
    return [m for _, m in missing]

print("[OK] 03_loaders · load_forest/load_grassland/check_presence carregados")


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
04_qa.py · QA de fechamento, def_sec e territórios.

Rode após 03_loaders.
"""
# ---------------------------------------------------------------------------
# QA de fechamento (agregações derivadas EM MEMÓRIA a partir dos n0)
# ---------------------------------------------------------------------------
def qa_fecha(df: pd.DataFrame, value_col: str, key_cols: list[str],
             area_col: str = "area_ha") -> list[str]:
    """Valida o fechamento das agregações derivadas das combinações por chave."""
    problems: list[str] = []
    if not len(df):
        return problems
    for key, grp in df.groupby(key_cols):
        total = grp[value_col].sum()
        tot_n1 = grp.groupby("n1", dropna=False)[value_col].sum().sum()
        tot_mark = grp.groupby("marcador", dropna=False)[value_col].sum().sum()
        if abs(tot_n1 - total) > CONF["MISSING_TOL"]:
            problems.append(f"{key}: n1 ({tot_n1:.1f}) ≠ total ({total:.1f})")
        if abs(tot_mark - total) > CONF["MISSING_TOL"]:
            problems.append(f"{key}: marcador ({tot_mark:.1f}) ≠ total "
                            f"({total:.1f})")
        if grp["marcador"].isna().any():
            problems.append(f"{key}: há classes sem marcador (legenda)")
        if (grp[value_col] < 0).any() or (grp[area_col] < 0).any():
            problems.append(f"{key}: valores negativos de {value_col}/área")
        dens = grp[value_col].sum() / max(grp[area_col].sum(), 1e-6)
        if grp[area_col].sum() > 0 and dens > CONF["DENSITY_MAX"]:
            problems.append(f"{key}: densidade média implícita {dens:.0f} "
                            f"t/ha > {CONF['DENSITY_MAX']} (unidades?)")
        if grp[area_col].sum() <= 0 and grp[value_col].sum() > 0:
            problems.append(f"{key}: estoque > 0 sem área")
    return problems


def qa_somas(df: pd.DataFrame, value_col: str, key_cols: list[str],
             area_col: str = "area_ha") -> list[str]:
    """QA de fechamento das agregações n0 (n1 = total, sem valores nulos)."""
    problems: list[str] = []
    if not len(df):
        return problems
    for key, grp in df.groupby(key_cols):
        total = grp[value_col].sum()
        tot_n1 = grp.groupby("n1", dropna=False)[value_col].sum().sum()
        tot_mark = grp.groupby("marcador", dropna=False)[value_col].sum().sum()
        if abs(tot_n1 - total) > CONF["MISSING_TOL"]:
            problems.append(f"{key}: n1 ({tot_n1:.1f}) ≠ total ({total:.1f})")
        if abs(tot_mark - total) > CONF["MISSING_TOL"]:
            problems.append(f"{key}: marcador ({tot_mark:.1f}) ≠ total "
                            f"({total:.1f})")
        if grp[value_col].isna().any():
            problems.append(f"{key}: há nulos em {value_col}")
        dens = grp[value_col].sum() / max(grp[area_col].sum(), 1e-6)
        if grp[area_col].sum() > 0 and dens > CONF["DENSITY_MAX"]:
            problems.append(f"{key}: densidade média implícita {dens:.0f} "
                            f"t/ha > {CONF['DENSITY_MAX']} (unidades?)")
        if grp[area_col].sum() <= 0 and grp[value_col].sum() > 0:
            problems.append(f"{key}: {value_col} > 0 sem área")
    return problems


def qa_defsec(df: pd.DataFrame,
              key_cols: tuple = ("version", "territorio")) -> list[str]:
    """QA da estratificação def_sec (1 linha por combinação classe × def_sec)."""
    problems: list[str] = []
    if not len(df) or "def_sec" not in df.columns:
        return problems
    # com filtro de estrato ativo, só resta 1 def_sec → relaxa as checagens
    # específicas de def_sec (duplicidade, rótulos, dominância de 'Não Observado')
    filtered = bool(CONF.get("DEF_SEC_FILTER"))
    if "def_sec_code" not in df.columns:
        df = df.assign(def_sec_code=df["def_sec"].map(defsec_code))
    vcols = [c for c in ("estoque_t", "biomassa_t", "anpp_t")
             if c in df.columns]
    for key, grp in df.groupby(list(key_cols) + ["classe"]):
        if "periodo" in grp.columns:
            partes = [pg for _, pg in grp.groupby(
                grp["periodo"].fillna("__anual__"), dropna=False)]
        else:
            partes = [grp]
        if not filtered:
            dups = sum(pg["def_sec"].duplicated().sum() for pg in partes)
            if dups:
                problems.append(f"{key}: {dups} combinações def_sec duplicadas "
                                "(classe × def_sec × mês)")
            unknown = grp[grp["def_sec_code"] < 0]
            if len(unknown):
                labels = ", ".join(sorted({str(v) for v in unknown["def_sec"]}))
                problems.append(f"{key}: rótulos def_sec fora da legenda: {labels}")
        if grp["area_ha"].isna().any() or (grp["area_ha"] < 0).any():
            problems.append(f"{key}: área nula/negativa em combinação def_sec")
        for vc in vcols:
            # só valida a coluna ATIVA do grupo (nas tabelas longas por produto
            # as demais colunas de valor ficam NaN por construção)
            if grp[vc].notna().any() and grp[vc].isna().any():
                problems.append(f"{key}: nulos em {vc}")
            if (grp[vc] < 0).any():
                problems.append(f"{key}: valores negativos em {vc}")
        if "natureza" in grp.columns and len(grp):
            nat = grp["natureza"].iloc[0]
            mark = grp["marcador"].iloc[0]
            if nat == "Antrópico" and mark != "outros":
                problems.append(f"{key}: natureza {nat} com marcador {mark}")
        if not filtered:
            no = grp[grp["def_sec_code"] == 0]["area_ha"].sum()
            a = grp["area_ha"].sum()
            if a > 0 and no / a > 0.5:
                problems.append(f"{key}: 'Não Observado' cobre {no / a:.0%} da área")
    return problems


def qa_territorios(df: pd.DataFrame, versions: list[str]) -> list[str]:
    """Avisa se alguma versão não aparecer nos CSVs."""
    found = sorted(df["version"].unique()) if "version" in df.columns else []
    missing = [v for v in versions if v not in found]
    return [f"versão sem arquivos: {m}" for m in missing]

print("[OK] 04_qa · QA carregado")


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
05_aggregations.py · Agregações em memória (colapso, pivôs, comparativos).

Rode após 04_qa.
"""
# ---------------------------------------------------------------------------
# Grassland: tabela principal LONGA por produto (biomass/carbon/anpp)
# ---------------------------------------------------------------------------
def expand_grass_product(df: pd.DataFrame) -> pd.DataFrame:
    """Deriva o produto semântico a partir do schema do grassland engine
    (principal ANUAL + mensais LONGOS).

    O CSV do GEE traz `produto='grassland'` em TODAS as linhas; aqui pivota
    em longo por produto de análise:
      - principal (sem `periodo`) → 'biomass' (`Biomassa acima do solo (t MS)`)
        E 'carbon' (`Carbono acima do solo (t C)`) — 2 linhas por combinação;
      - biomassa MENSAL descartada (o anual já cobre o comparativo);
      - anpp MENSAL → 'anpp' (soma dos meses feita por por_classe).
    """
    if df is None or not len(df):
        return pd.DataFrame()
    df = df.copy()
    if "periodo" not in df.columns:
        df["periodo"] = pd.NA
    annual = df[df["periodo"].isna()].copy()
    monthly = df[df["periodo"].notna()].copy()
    longa = []
    if len(annual):
        for p, active_col in (("biomass", "biomassa_t"),
                              ("carbon", "estoque_t")):
            if active_col not in annual.columns:
                continue
            part = annual.copy()
            part["produto"] = p
            for c in ("estoque_t", "biomassa_t", "anpp_t"):
                if c in part.columns and c != active_col:
                    part[c] = pd.NA
            longa.append(part)
    if len(monthly):
        anpp = monthly[monthly["anpp_t"].notna()].copy()
        if len(anpp):
            anpp["produto"] = "anpp"
            for c in ("estoque_t", "biomassa_t"):
                if c in anpp.columns:
                    anpp[c] = pd.NA
            longa.append(anpp)
    if not longa:
        return pd.DataFrame()
    out = pd.concat(longa, ignore_index=True)
    return out[out["produto"].isin(["biomass", "carbon", "anpp"])]


# ---------------------------------------------------------------------------
# Colapso de combinações (classe × def_sec; e meses) em UMA linha por classe.
#   área/valor = soma · incerteza = QUADRATURA (sqrt Σ ι²)
# ---------------------------------------------------------------------------
_PART_KEYS = ("produto", "versao", "version", "territorio", "Bioma", "UF",
              "classe", "n0", "n1", "n2", "n3", "n4", "marcador", "natureza",
              "ano")
_SUM_COLS = ("area_ha", "estoque_t", "biomassa_t", "anpp_t")


def por_classe(df: "pd.DataFrame") -> "pd.DataFrame":
    """Colapsa as linhas de combinação (classe × def_sec; × mês) em UMA linha
    por classe: área/valores somam; incerteza em quadratura."""
    if df is None or not len(df):
        return df
    keys = [k for k in _PART_KEYS if k in df.columns]
    cols = [c for c in _SUM_COLS if c in df.columns]
    out = df.groupby(keys, dropna=False)[cols].sum().reset_index()
    for c in ("unidade",):
        if c in df.columns and c not in keys:
            out[c] = (df.groupby(keys, dropna=False)[c]
                      .first().reset_index(drop=True))
    if "incerteza_t" in df.columns:
        inc = (df.groupby(keys, dropna=False)["incerteza_t"]
               .apply(lambda s: np.sqrt(np.sum(
                   np.square(pd.to_numeric(s, errors="coerce").fillna(0)))))
               .rename("incerteza_t")
               .reset_index())
        out = out.drop(columns=["incerteza_t"], errors="ignore")
        out = out.merge(inc, on=keys, how="left")
    return out


# ---------------------------------------------------------------------------
# Pivôs
# ---------------------------------------------------------------------------
def pivot_marcador(df: pd.DataFrame, value_col: str,
                   keys=("version",)) -> pd.DataFrame:
    """Pivô marcador (florestal/campestre/outros) × valor, somando os n0."""
    rows = df.groupby(list(keys) + ["marcador"], dropna=False)[value_col].sum()
    return rows.reset_index()


def pivot_n0(df: pd.DataFrame, value_col: str,
             keys=("version",)) -> pd.DataFrame:
    """Tabela de classes (col × versão) com n0 e n2."""
    rows = df.copy()
    rows["n0"] = rows["classe"].apply(n0_label)
    rows["n2"] = rows["classe"].apply(n2_of)
    return rows.pivot_table(index=["n0", "n2"], columns=list(keys),
                            values=value_col, aggfunc="sum").reset_index()


def total_linha(df: pd.DataFrame, value_col: str, keys=("version",)) -> pd.DataFrame:
    """Total (n0) por chave × território — sempre derivado, nunca exportado."""
    return (df.groupby(list(keys) + ["territorio"])[value_col].sum()
            .reset_index())


# ---------------------------------------------------------------------------
# Comparativos OFFLINE
# ---------------------------------------------------------------------------
def build_compare_forest(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela version × marcador (florestal/campestre/outros + total)."""
    if df is None or not len(df):
        return pd.DataFrame()
    rows = df.groupby(["version", "marcador"], as_index=False).agg(
        area_ha=("area_ha", "sum"),
        estoque_t=("estoque_t", "sum"),
        incerteza_t=("incerteza_t", "sum"))
    tot = df.groupby(["version"], as_index=False).agg(
        area_ha=("area_ha", "sum"),
        estoque_t=("estoque_t", "sum"),
        incerteza_t=("incerteza_t", "sum"))
    tot["marcador"] = "total"
    out = pd.concat([rows, tot], ignore_index=True)
    out.insert(1, "ano", CONF["YEAR"])
    return out[["version", "ano", "marcador", "area_ha", "estoque_t",
                "incerteza_t"]]


GRASS_VALUE_COL = {"biomass": "biomassa_t", "carbon": "estoque_t",
                   "anpp": "anpp_t"}
GRASS_PRODUCT_LABEL = {
    "biomass": "Biomassa acima do solo (t MS)",
    "anpp": "Produtividade primária líquida acima do solo (t MS/ano)",
    "carbon": "Carbono acima do solo (t C)"}

# Rótulo (nome da COLUNA acordada) por coluna interna de valor.
VALUE_LABEL = {
    "estoque_t": "Carbono acima do solo (t C)",
    "biomassa_t": "Biomassa acima do solo (t MS)",
    "anpp_t": "Produtividade primária líquida acima do solo (t MS/ano)",
}
# Idem, para DENSIDADE (t/ha) — usado nos gráficos de média ponderada.
VALUE_DENS_LABEL = {
    "estoque_t": "Carbono acima do solo (t C/ha)",
    "biomassa_t": "Biomassa acima do solo (t MS/ha)",
    "anpp_t": "Produtividade primária líquida acima do solo (t MS/ha/ano)",
}


def grass_value_col(produto) -> str:
    """Coluna de valor do produto campestre no consolidado grassland."""
    return GRASS_VALUE_COL.get(str(produto), "biomassa_t")


def dens_ponderada(df: pd.DataFrame, value_col: str,
                   keys=("classe",)) -> pd.DataFrame:
    """Densidade ponderada por área: `sum(valor) / sum(area_ha)` por chave."""
    cols = list(keys) + ["area_ha", "valor", "dens"]
    if df is None or not len(df):
        return pd.DataFrame(columns=cols)
    g = (df.groupby(list(keys), dropna=False)
         .agg(area_ha=("area_ha", "sum"), valor=(value_col, "sum"))
         .reset_index())
    g["dens"] = np.where(g["area_ha"] > 0, g["valor"] / g["area_ha"], np.nan)
    return g


def build_compare_grassland(df: pd.DataFrame) -> pd.DataFrame:
    """Tabela produto × classe (+ TOTAL) da versão de referência."""
    cols = ["produto", "versao", "ano", "territorio", "unidade", "classe",
            "n0", "n1", "marcador", "area_ha", "valor_t"]
    if df is None or not len(df):
        return pd.DataFrame(columns=cols)
    ref = str(CONF["GRASSLAND_REF_VERSION"])
    d = df[df["versao"].astype(str) == ref].copy() if "versao" in df.columns else df.copy()
    if not len(d):
        return pd.DataFrame(columns=cols)
    out = []
    for p in ("biomass", "carbon", "anpp"):
        vc = grass_value_col(p)
        sub = d[d["produto"] == p].copy()
        if not len(sub):
            continue
        sub = por_classe(sub)
        if not len(sub):
            continue
        dec = pd.DataFrame({
            "produto": p, "versao": ref, "ano": CONF["YEAR"],
            "territorio": sub["territorio"].iloc[0]
            if "territorio" in sub else "",
            "unidade": sub["unidade"].iloc[0] if "unidade" in sub else "t",
            "classe": sub["classe"], "n0": sub["n0"], "n1": sub["n1"],
            "marcador": sub["marcador"], "area_ha": sub["area_ha"],
            "valor_t": sub[vc]})
        tot = pd.DataFrame([{
            "produto": p, "versao": ref, "ano": CONF["YEAR"],
            "territorio": dec["territorio"].iloc[0],
            "unidade": dec["unidade"].iloc[0],
            "classe": "", "n0": "TOTAL", "n1": "—", "marcador": "total",
            "area_ha": sub["area_ha"].sum(), "valor_t": sub[vc].sum()}])
        out.append(pd.concat([dec, tot], ignore_index=True))
    return pd.concat(out, ignore_index=True) if out else pd.DataFrame(columns=cols)


# ---------------------------------------------------------------------------
# Coluna `data` (YYYY-MM-DD) — compatível com o Looker Studio
# ---------------------------------------------------------------------------
def add_data_col(df: pd.DataFrame) -> pd.DataFrame:
    """Adiciona a coluna `data` (ISO `YYYY-MM-DD`) para o Looker Studio.

    - linhas mensais (coluna `periodo` = 'YYYY_MM') → 'YYYY-MM-01';
    - demais → 'YYYY-01-01' (de `ano`/`ano_lulc`/CONF['YEAR']).
    """
    if df is None or not len(df):
        return df
    df = df.copy()
    y = CONF["YEAR"]

    def _ano01(v):
        return f"{int(v):04d}-01-01" if pd.notna(v) else f"{int(y):04d}-01-01"

    if "periodo" in df.columns:
        def _mes(p):
            try:
                yy, mm = str(p).split("_")[:2]
                return f"{int(yy):04d}-{int(mm):02d}-01"
            except (ValueError, AttributeError):
                return f"{int(y):04d}-01-01"
        per = df["periodo"]
        if "ano" in df.columns:
            base = df["ano"].map(_ano01)
        elif "ano_lulc" in df.columns:
            base = df["ano_lulc"].map(_ano01)
        else:
            base = f"{int(y):04d}-01-01"
        df["data"] = [(_mes(p) if pd.notna(p) else b)
                      for p, b in zip(per, base if hasattr(base, "__iter__")
                                      else [base] * len(df))]
    else:
        if "ano" in df.columns:
            df["data"] = df["ano"].map(_ano01)
        elif "ano_lulc" in df.columns:
            df["data"] = df["ano_lulc"].map(_ano01)
        else:
            df["data"] = f"{int(y):04d}-01-01"
    return df

print("[OK] 05_aggregations · agregações carregadas")


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — cells_for_collab_notebook
========================================================================
06_references.py · Catálogo/mapa das REFERÊNCIAS + análise T1/T2.

As estatísticas das referências são computadas NO EARTH ENGINE (run/run_references.js)
e exportadas **1 CSV por FAMÍLIA** (`stats_references_<grupo>_<ano>.csv`), com as
métricas de cada ref em COLUNAS descritivas. Aqui:
  · `load_references_csv` faz o *unpivot* das colunas nativas para longo,
    derivando (termo, unidade, fator) do NOME da coluna;
  · T1 (`cross_2023`) e T2 (`best_reference_year`) cruzam por TERMO.

Rode após 05_aggregations.
"""
# ---------------------------------------------------------------------------
# Constantes e mapas
# ---------------------------------------------------------------------------
CARBON_FRACTION = CONF["CARBON_FRACTION"]  # AGB (t) -> C (t)
REF_MIN_VALID = CONF["REF_MIN_VALID"]      # fração mínima de área válida

# Família (grupo do arquivo) → rótulo + época (o ano-base vai no nome do grupo).
GRUPO_META = {
    "saatchi":   {"group": "Saatchi 2011", "epoch": "2011"},
    "baccini":   {"group": "Baccini 2012", "epoch": "2012"},
    "zimbres":   {"group": "Zimbres 2021", "epoch": "2019"},
    "avitabile": {"group": "Avitabile 2016", "epoch": "2010"},
    "xu":        {"group": "Xu & Saatchi 2021", "epoch": "2019"},
    "tcn":       {"group": "TCN (3ª CN)", "epoch": "baseline"},
    "qcn":       {"group": "QCN (4ª CN)", "epoch": "baseline"},
    "qcnr":      {"group": "QCN-r (4ª CN)", "epoch": "1985"},
}

# Colunas NATIVAS das referências (nome → termo). As colunas com `*` (derivadas
# no estágio de estatística) são ignoradas — o carbono sai de AGB × fator.
REF_NATIVE_COLS = {
    "Biomassa acima do solo (t MS)": "agb",
    "Carbono acima do solo (t C)": "agc",
    "Biomassa abaixo do solo (t MS)": "bgb",
    "Carbono abaixo do solo (t C)": "bgc",
    "Carbono na madeira morta (t C)": "cdw",
    "Carbono na serrapilheira (t C)": "litter",
    "Carbono vivo (t C)": "live",
    "Carbono total (t C)": "total",
    "Incerteza (t C)": "error",
}

# termo → (magnitude, factor_to_c, unidade nativa)
REF_MAG = {
    "agb": ("biomassa", CARBON_FRACTION, "t MS/ha"),
    "bgb": ("biomassa", CARBON_FRACTION, "t MS/ha"),
    "agc": ("carbono", 1.0, "t C/ha"),
    "bgc": ("carbono", 1.0, "t C/ha"),
    "cdw": ("carbono", 1.0, "t C/ha"),
    "litter": ("carbono", 1.0, "t C/ha"),
    "live": ("carbono", 1.0, "t C/ha"),
    "total": ("carbono", 1.0, "t C/ha"),
    "error": ("incerteza", 1.0, "t C/ha"),
}

# Termos elegíveis ao "melhor ano de referência" (T2).
T2_TERMS = ("agb", "agc", "total", "live")


def load_references_csv(lulc_year: int = 2023,
                        territorio: str | None = None) -> "pd.DataFrame":
    """Lê os CSVs agrupados `stats_references_*_<ano>.csv`, filtra o território
    e monta o DataFrame LONGO com UMA linha por (ref × classe).

    Colunas: ref_id, ref_family, band, year, classe, area_ha, total_native,
    dens_native, ref_term, unit, factor_to_c, epoch, group, variable,
    t2_candidate.
    """
    territorio = territorio or CONF.get("REF_TERRITORIO", "Pantanal (bioma)")
    files = sorted(csv_dir().glob(f"stats_references_*_{lulc_year}.csv"))
    if not files:
        raise FileNotFoundError(
            f"nenhum stats_references_*_{lulc_year}.csv em {csv_dir()} — "
            "rode o run/run_references.js no GEE e re-exporte para o Drive.")
    frames = []
    for fp in files:
        grupo = fp.stem.replace("stats_references_", "").replace(
            f"_{lulc_year}", "")
        meta = GRUPO_META.get(grupo, {"group": grupo, "epoch": "?"})
        df = _derive_territorio(pd.read_csv(fp, encoding="utf-8"))
        if "Desmatamento e vegetação secundária" in df.columns:
            df["def_sec"] = df["Desmatamento e vegetação secundária"]
        df = _filter_defsec(df)
        if "territorio" in df.columns:
            df = df[df["territorio"] == territorio]
        if not len(df):
            continue
        cols = [c for c in REF_NATIVE_COLS if c in df.columns]
        if not cols:
            print(f"[AVISO] {fp.name}: nenhuma coluna de métrica reconhecida.")
            continue
        long = df[["classe", "Área (ha)"] + cols].melt(
            id_vars=["classe", "Área (ha)"], value_vars=cols,
            var_name="col", value_name="total_native")
        long["ref_family"] = grupo
        long["ref_term"] = long["col"].map(REF_NATIVE_COLS)
        long["ref_id"] = grupo + "_" + long["ref_term"]
        long["unit"] = long["ref_term"].map(lambda t: REF_MAG[t][2])
        long["factor_to_c"] = long["ref_term"].map(lambda t: REF_MAG[t][1])
        long["group"] = meta["group"]
        long["epoch"] = meta["epoch"]
        long["t2_candidate"] = long["ref_term"].isin(T2_TERMS)
        long["band"] = ""
        long["year"] = None
        long["variable"] = long["ref_term"]
        long = long.rename(columns={"Área (ha)": "area_ha"})
        frames.append(long.drop(columns=["col"]))
    if not frames:
        return pd.DataFrame()
    df = pd.concat(frames, ignore_index=True)
    for c in ("classe", "area_ha", "total_native"):
        df[c] = pd.to_numeric(df[c], errors="coerce")
    df = df.dropna(subset=["classe"])
    df = (df.groupby(["ref_id", "ref_family", "band", "classe"], dropna=False)
          .agg(area_ha=("area_ha", "sum"),
               total_native=("total_native", "sum"),
               year=("year", "first"), ref_term=("ref_term", "first"),
               unit=("unit", "first"), factor_to_c=("factor_to_c", "first"),
               epoch=("epoch", "first"), group=("group", "first"),
               variable=("variable", "first"),
               t2_candidate=("t2_candidate", "first"))
          .reset_index())
    df["area_ha"] = df["area_ha"].round(1)
    # sentinela de nodata (ex.: TCN ~ ±9.22e17) → NaN
    df.loc[df["total_native"].abs() >= 1e15, "total_native"] = np.nan
    df["total_native"] = df["total_native"].round(1)
    df["dens_native"] = np.where(df["area_ha"] > 0,
                                 df["total_native"] / df["area_ha"], 0.0)
    df["dens_native"] = df["dens_native"].round(3)
    return df


# ---------------------------------------------------------------------------
# Termos comuns: valor da referência na unidade do produto
# ---------------------------------------------------------------------------
ALLOWED_TERMS = {
    "agc":     ("agc", "live"),
    "carbon":  ("agc", "live", "total", "bgc", "bgb", "cdw", "litter"),
    "biomass": ("agb",),
}
PRODUCT_TERM = {"forest": "agc", "grassland_carbon": "carbon",
                "grassland_biomass": "biomass"}


def ref_value_for_product(df: "pd.DataFrame", product_term: str) -> "pd.DataFrame":
    """Adiciona `ref_value` (densidade na unidade do produto) e `val_unit`."""
    df = df.copy()
    if product_term == "biomass":
        df["ref_value"] = df["dens_native"]
        df["val_unit"] = df["unit"]
    else:
        df["ref_value"] = df["dens_native"] * df["factor_to_c"]
        df["val_unit"] = "t C/ha"
    df["ref_total"] = df["total_native"] * df["factor_to_c"]
    return df


def product_lines(forest: "pd.DataFrame", grassland: "pd.DataFrame",
                  year: int = 2023) -> "pd.DataFrame":
    """Linhas produto×versão×classe com densidade e incerteza (ano)."""
    rows = []
    if forest is not None and len(forest):
        f = forest[forest["ano"] == year] if "ano" in forest else forest
        f = por_classe(f)
        inc = f["incerteza_t"] if "incerteza_t" in f.columns else pd.Series(
            pd.NA, index=f.index)
        rows.append(pd.DataFrame({
            "produto": "forest", "versao": f["version"].astype(str),
            "classe": pd.to_numeric(f["classe"], errors="coerce"),
            "n0": f["n0"], "n1": f["n1"], "natureza": f["natureza"],
            "marcador": f["marcador"], "area_ha": f["area_ha"],
            "dens_prod": f["estoque_t"] / f["area_ha"].replace(0, pd.NA),
            "inc_tcha": inc / f["area_ha"].replace(0, pd.NA),
            "term": "agc"}))
    if grassland is not None and len(grassland):
        g = grassland[grassland["ano"] == year] if "ano" in grassland else grassland
        g = g[g["produto"].isin(["carbon", "biomass"])].copy()
        for pname, pproduto, vcol in (("grassland_carbon", "carbon", "estoque_t"),
                                      ("grassland_biomass", "biomass", "biomassa_t")):
            sub = g[g["produto"] == pproduto].copy()
            if "periodo" in sub.columns:
                sub = sub[sub["periodo"].isna()]
            sub = por_classe(sub)
            rows.append(pd.DataFrame({
                "produto": pname, "versao": sub["versao"].astype(str),
                "classe": pd.to_numeric(sub["classe"], errors="coerce"),
                "n0": sub["n0"], "n1": sub["n1"], "natureza": sub["natureza"],
                "marcador": sub["marcador"], "area_ha": sub["area_ha"],
                "dens_prod": sub[vcol] / sub["area_ha"].replace(0, pd.NA),
                "inc_tcha": pd.NA, "term": pproduto}))
    df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    if len(df):
        df = df.dropna(subset=["classe", "dens_prod"])
    return df


def cross_2023(products: "pd.DataFrame", refs: "pd.DataFrame",
               ref_epoch_default: dict | None = None) -> "pd.DataFrame":
    """T1 · linha por produto×versão×classe×referência (época representativa)."""
    if not len(products) or not len(refs):
        return pd.DataFrame()
    frames = []
    for term, gterm in products.groupby("term"):
        allowed = ALLOWED_TERMS.get(term, ())
        sub = refs[refs["ref_term"].isin(allowed)].copy()
        if not len(sub):
            continue
        sub = _representative_rows(sub)
        j = gterm.merge(sub, on="classe", how="left")
        j = ref_value_for_product(j, term)
        j["diff"] = j["dens_prod"] - j["ref_value"]
        j["diff_pct"] = 100 * j["diff"] / j["ref_value"].replace(0, pd.NA)
        j["val_pct"] = j["area_ha_y"] / j["area_ha_x"].replace(0, pd.NA)
        j["dentro_inc"] = False
        m_inc = j["diff"].notna() & j["inc_tcha"].notna()
        j.loc[m_inc, "dentro_inc"] = (j.loc[m_inc, "diff"].abs()
                                      <= j.loc[m_inc, "inc_tcha"])
        frames.append(j)
    out = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    cols = ["produto", "versao", "classe", "n0", "n1", "natureza", "marcador",
            "area_ha_x", "dens_prod", "inc_tcha", "ref_id", "group", "variable",
            "term", "ref_term", "unit", "val_unit", "epoch", "year", "band",
            "area_ha_y", "val_pct", "dens_native", "ref_value", "ref_total",
            "diff", "diff_pct", "dentro_inc"]
    if len(out):
        out = out[cols]
        cols = ["produto", "versao", "classe", "n0", "n1",
                "natureza", "marcador", "area_cls_ha", "dens_prod", "inc_tcha",
                "term", "ref_id", "ref_group", "ref_variable", "ref_term",
                "ref_unit", "val_unit", "ref_epoch", "ref_year", "ref_band",
                "ref_area_ha", "ref_val_pct", "ref_dens_native",
                "ref_value", "ref_total", "diff_dens", "diff_pct",
                "dentro_incerteza"]
        out = out.rename(columns={"area_ha_x": "area_cls_ha", "group": "ref_group",
                        "variable": "ref_variable",
                        "unit": "ref_unit", "epoch": "ref_epoch",
                        "year": "ref_year", "band": "ref_band",
                        "area_ha_y": "ref_area_ha", "val_pct": "ref_val_pct",
                        "dens_native": "ref_dens_native",
                        "diff": "diff_dens", "dentro_inc": "dentro_incerteza"})
        out = out[cols]
    return out


def _representative_rows(refs: "pd.DataFrame") -> "pd.DataFrame":
    """Mantém 1 linha por ref×classe (banda/época representativa)."""
    out = []
    for (rid, cls), g in refs.groupby(["ref_id", "classe"]):
        if "year" in g.columns and g["year"].notna().any():
            g = g[g["year"] == g["year"].max()]
        else:
            g = g.head(1)
        out.append(g)
    return pd.concat(out, ignore_index=True) if out else refs


def best_reference_year(products: "pd.DataFrame", refs: "pd.DataFrame",
                        min_valid: float = REF_MIN_VALID) -> "pd.DataFrame":
    """T2 · por produto×versão×classe: ref+época que minimiza |dens|."""
    if not len(products) or not len(refs):
        return pd.DataFrame()
    rows = []
    for (term, prod), (pid, versao, cls), prodrow in _iter_products(products):
        allowed = ALLOWED_TERMS.get(term, ())
        sub = refs[(refs["ref_term"].isin(allowed)) & (refs["t2_candidate"])]
        sub = sub[sub["classe"] == cls].copy()
        if not len(sub):
            rows.append(_t2_row(prodrow, None))
            continue
        sub = ref_value_for_product(sub, term)
        sub = sub.dropna(subset=["ref_value"])
        if not (prodrow["area_ha"] and prodrow["area_ha"] > 0):
            rows.append(_t2_row(prodrow, None))
            continue
        sub["val_pct"] = sub["area_ha"] / prodrow["area_ha"]
        sub = sub[sub["val_pct"] >= min_valid]
        if not len(sub):
            rows.append(_t2_row(prodrow, None))
            continue
        sub["diff_abs"] = (prodrow["dens_prod"] - sub["ref_value"]).abs()
        best = sub.sort_values("diff_abs").iloc[0]
        top = list(sub.sort_values("diff_abs")[["ref_id", "epoch"]]
                   .itertuples(index=False, name=None))
        rows.append(_t2_row(prodrow, best, top))
    df = pd.DataFrame(rows)
    return df


def _iter_products(products: "pd.DataFrame"):
    for (pid, versao, cls), g in products.groupby(["produto", "versao", "classe"]):
        row = g.iloc[0]
        yield (row["term"], ""), (pid, versao, cls), {
            "produto": pid, "versao": versao, "classe": cls, "n0": row["n0"],
            "n1": row["n1"], "natureza": row["natureza"],
            "marcador": row["marcador"], "area_ha": row["area_ha"],
            "dens_prod": row["dens_prod"], "term": row["term"]}


def _clean_year(year):
    try:
        y = float(year)
    except (TypeError, ValueError):
        return None
    return str(int(y)) if y == y else None


def _t2_row(prow, best, top=None):
    if best is None:
        return {**{k: prow[k] for k in ("produto", "versao", "classe", "n0",
                                        "n1", "natureza", "marcador", "area_ha",
                                        "dens_prod")},
                "melhor_ref": None, "melhor_epoch": None, "melhor_year": None,
                "melhor_dens_ref": None, "diff_abs": None, "diff_pct": None,
                "fator_impl": None, "candidatos_top3": None,
                "recomendacao": "sem referência confiável (cobertura/termo)"}
    rec = best["ref_id"] + "/" + (_clean_year(best.get("year")) or best["epoch"])
    dens = float(best["ref_value"])
    diff_abs = abs(float(prow["dens_prod"]) - dens)
    return {**{k: prow[k] for k in ("produto", "versao", "classe", "n0", "n1",
                                    "natureza", "marcador", "area_ha",
                                    "dens_prod")},
            "melhor_ref": best["ref_id"], "melhor_epoch": best["epoch"],
            "melhor_year": best.get("year"),
            "melhor_dens_ref": round(dens, 3),
            "diff_abs": round(diff_abs, 3),
            "diff_pct": round(100 * diff_abs / max(float(prow["dens_prod"]), 1e-9), 1),
            "fator_impl": round(float(prow["dens_prod"]) / dens, 3)
            if dens > 0 else None,
            "candidatos_top3": " | ".join(f"{r}/{e}" for r, e in (top or [])[:3]),
            "recomendacao": rec}


def qa_references(refs: "pd.DataFrame", min_valid: float = REF_MIN_VALID) -> list[str]:
    problems = []
    if not len(refs):
        return problems
    tot = refs.groupby(["ref_id", "band", "classe"])["area_ha"].sum().reset_index()
    for (rid, band), g in tot.groupby(["ref_id", "band"]):
        zero = (g["area_ha"] == 0).sum()
        if zero:
            problems.append(f"{rid}/{band}: {zero} classes com área 0 "
                            "(ref não cobre — esperado em campestre p/ AGB)")
    return problems

print("[OK] 06_references · catálogo de famílias + load_references_csv + T1/T2")


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 10_auth_drive.py
===============================================================================
Etapa 10 · monta o Google Drive e configura CONF (paths de CSV/figuras/saída).

Rode APÓS 00_config..06_references. As estatísticas (produtos E referências)
são computadas NO GEE (run/run_pantanal.js / run/run_references.js) e apenas
LIDAS aqui — este notebook NÃO autentica nem inicializa o Earth Engine
(exceção: a etapa opcional 16_maps).
"""
print("=" * 68)
print("statistics · 10_auth_drive · autenticação Drive (sem Earth Engine)")
print("=" * 68)

# 1) Montagem do Google Drive
from google.colab import drive  # noqa: E402

drive.mount("/content/drive")
print("[DRIVE] montado em /content/drive")

# 2) Configuração da pasta de output (CSVs + figuras + relatório)
DRIVE_ROOT = "/content/drive/MyDrive/mapbiomas_carbono/pantanal"

import sys  # noqa: E402
from pathlib import Path  # noqa: E402

# A pasta das células é adicionada ao sys.path para rodar as etapas com %run
# apontando para os .py (os componentes já estão no namespace — 00..06).
GCN_DIR = Path(DRIVE_ROOT) / "cells_for_collab_notebook"
if str(GCN_DIR) not in sys.path:
    sys.path.insert(0, str(GCN_DIR))
if str(DRIVE_ROOT) not in sys.path:
    sys.path.insert(0, DRIVE_ROOT)
print(f"[PATH] sys.path += {GCN_DIR} / {DRIVE_ROOT}")

CONF["DRIVE_ROOT"] = DRIVE_ROOT
CONF["CSV_DIR"] = "estatisticas"      # CSVs do run/run_pantanal.js / run_references.js
CONF["FIG_DIR"] = "figures"           # biblioteca de figuras
CONF["OUT_DIR"] = "relatorio"         # XLSX + .md + consolidados

print()
print("[OK] Preparado. Pasta de output:", CONF["DRIVE_ROOT"])
print("     Próximas: 11_consolidate → 12_join_largo → 13_figures → "
      "14_report → 15_references → 16_maps (opcional).")
print("=" * 68)

In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 11_consolidate.py
========================================================================
Etapa 11 · Consolida os CSVs do GEE (LONGO canônico) e valida somas.

Rode após 10_auth_drive. Usa CONF + funções de 00..06 no namespace.

  1. checa arquivos esperados (indica qual driver rodar no GEE quando faltar)
  2. consolida forest (todas as versões presentes) e grassland
  3. QA de somas (n0 = n1 = total) e territórios por versão
  4. salva consolidados LONGO em OUT_DIR (consolidado_forest_2023.csv e
     consolidado_grassland_2023.csv) — canônicos p/ QA e 15_references
"""
print("=" * 68)
print("statistics · 11_consolidate · consolidação + QA")
print("=" * 68)

_f = CONF.get("DEF_SEC_FILTER")
print("[FILTRO] def_sec =", ", ".join(_f) if _f else "todos")

missing = check_presence()

forest = add_data_col(load_forest())
grassland = add_data_col(expand_grass_product(load_grassland()))

print()
print(f"[FOREST] {len(forest)} linhas · versões:",
      sorted(forest["version"].unique()) if len(forest) else "—")
print(f"[GRASSLAND] {len(grassland)} linhas (longo por produto) · produtos:",
      sorted(grassland["produto"].unique()) if len(grassland) else "—")

# --- comparativos OFFLINE (a partir do consolidado) -------------------------
cmpf = build_compare_forest(forest)
cmpg = build_compare_grassland(grassland)
print(f"[COMPARE] forest {len(cmpf)} · grassland {len(cmpg)} linhas (offline)")

# --- QA forest ---------------------------------------------------------------
if len(forest):
    problems = qa_somas(forest, "estoque_t", ["version", "territorio", "UF"])
    problems += qa_defsec(forest, ("version", "territorio", "UF"))
    problems += qa_territorios(forest, CONF["FOREST_VERSIONS"])
    for p in problems:
        print("[QA !]", p)
    if not problems:
        print("[QA OK] forest: n1/marcador = total em todas as versões/"
              "territórios e combinações def_sec coerentes")
    out = out_dir() / "consolidado_forest_2023.csv"
    forest.to_csv(out, index=False)
    print(f"[OUT] {out}")

# --- QA grassland ------------------------------------------------------------
if len(grassland):
    problems = []
    for p in grassland["produto"].unique():
        vc = grass_value_col(p)
        problems += qa_somas(grassland[grassland["produto"] == p], vc,
                             ["produto", "versao", "territorio", "UF"])
    problems += qa_defsec(grassland, ("produto", "versao", "territorio", "UF"))
    for p in problems:
        print("[QA !]", p)
    if not problems:
        print("[QA OK] grassland: n1/marcador = total em todos os produtos"
              " e combinações def_sec coerentes")
    out = out_dir() / "consolidado_grassland_2023.csv"
    grassland.to_csv(out, index=False)
    print(f"[OUT] {out}")

if missing:
    print()
    print("[ATENÇÃO] faltam arquivos — rode no GEE os steps indicados e faça o")
    print("          download/export dos CSVs para", csv_dir(),
          "depois rode a 11_consolidate novamente.")
print()
print(f"[ARTEFATOS] {len(manifest())} gerados")
print("=" * 68)

In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 12_join_largo.py
========================================================================
Etapa 12 · Junta (join) as tabelas exportadas SEPARADAS pelo GEE numa única
tabela LARGA: uma linha por (Bioma, UF, classe, def_sec) e uma coluna por
métrica. Usa CONF + funções de 00..06 no namespace.

Fontes:
  · forest      → CSV largo (colunas '<vXX> Biomassa acima do solo (t MS)' /
                  '<vXX> Carbono acima do solo (t C)*') → é a BASE da tabela;
  · grassland   → 1 CSV por versão (principal anual) → colunas
                  'grassland <v> Biomassa acima do solo (t MS)' /
                  '... Carbono acima do solo (t C)';
  · references  → 1 CSV por FAMÍLIA (colunas de métrica descritivas, ex.:
                  'Biomassa acima do solo (t MS)', 'Carbono acima do solo (t C)*').

O join usa a chave (Bioma, UF, classe, def_sec). Saída:
  OUT_DIR/consolidado_largo_2023.csv
"""
print("=" * 68)
print("statistics · 12_join_largo · join largo")
print("=" * 68)

import re
import numpy as np
import pandas as pd

KEY = ["Bioma", "UF", "classe", "def_sec"]
CATS = ["cobertura (n0)", "cobertura (n1)", "cobertura (n2)",
        "cobertura (n3)", "cobertura (n4)"]


def _read(pattern):
    return [pd.read_csv(f) for f in sorted(csv_dir().glob(pattern))]


def _prep(df):
    df = df.copy()
    if "Desmatamento e vegetação secundária" in df.columns:
        df["def_sec"] = df["Desmatamento e vegetação secundária"].astype(str)
    for c in ("Bioma", "UF"):
        if c in df.columns:
            df[c] = df[c].astype(str)
    if "classe" in df.columns:
        df["classe"] = pd.to_numeric(df["classe"], errors="coerce")
    # sentinela de nodata (ex.: TCN/grassland ~ ±9.22e17) → NaN
    num = df.select_dtypes(include=[np.number]).columns
    df[num] = df[num].mask(df[num].abs() >= 1e15)
    return _filter_defsec(df)


# --- base: forest largo ------------------------------------------------------
ff = _read("stats_forest_*2023*.csv")
if not ff:
    raise FileNotFoundError("nenhum stats_forest_*2023*.csv em " + str(csv_dir()))
base = _prep(pd.concat(ff, ignore_index=True))
metric_cols = [c for c in base.columns if re.match(r"^v\d+\s+", str(c))]
base = base[KEY + CATS + ["Área (ha)"] + metric_cols].copy()
print(f"[BASE] forest largo: {len(base)} linhas · {len(metric_cols)} colunas de versão")

parts = [base]

# --- grassland principal (sem `período`) -------------------------------------
for df in _read("stats_grassland_*2023*.csv"):
    if "período" in df.columns:
        continue  # mensais ficam de fora da tabela larga
    df = _prep(df)
    v = str(df["versao"].iloc[0]) if ("versao" in df.columns and len(df)) else "vXX"
    ren = {}
    for c in ("Biomassa acima do solo (t MS)", "Carbono acima do solo (t C)"):
        if c in df.columns:
            ren[c] = f"grassland {v} {c}"
    df = df.rename(columns=ren)
    keep = KEY + [c for c in ren.values() if c in df.columns]
    parts.append(df[keep])
    print(f"[GRASS] {v}: {len(df)} linhas ·", list(ren.values()))

# --- references (1 CSV por família; colunas de métrica descritivas) ----------
NON_METRIC = set(KEY + CATS + [
    "ecossistema", "Unidade Federativa",
    "Desmatamento e vegetação secundária", "ano_lulc", "Área (ha)"])
for fp in sorted(csv_dir().glob("stats_references_*2023*.csv")):
    df = _prep(pd.read_csv(fp))
    grupo = fp.stem.replace("stats_references_", "").replace("_2023", "")
    mcols = [c for c in df.columns if c not in NON_METRIC]
    ren = {c: f"{grupo} {c}" for c in mcols}
    df = df.rename(columns=ren)
    parts.append(df[[c for c in (KEY + [ren[c] for c in mcols]) if c in df.columns]])
    print(f"[REF] {grupo}: {len(df)} linhas · {len(mcols)} métrica(s)")

# --- join largo --------------------------------------------------------------
out = parts[0]
for p in parts[1:]:
    out = out.merge(p, on=KEY, how="outer")
out = out.sort_values(KEY).reset_index(drop=True)
out = add_data_col(out)  # coluna `data` (YYYY-MM-DD) p/ Looker Studio

path = out_dir() / "consolidado_largo_2023.csv"
out.to_csv(path, index=False)
print(f"[OUT] {path} · {out.shape[0]} linhas × {out.shape[1]} colunas")
print("=" * 68)


In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 13_figures.py
==================================================================
Etapa 13 · Biblioteca de figuras das estatísticas. Rode após 11_consolidate.

Figuras (salvas na MESMA pasta de output no Drive, FIG_DIR):
  f1_forest_estoque_por_classe.png    estoque (t) por classe C11, por versão
  f2_forest_estoque_por_marcador.png  florestal × campestre × outros, por versão
  f3_forest_compare_versoes.png       total e florestal por versão (v03→v08)
  f4_grassland_produtos.png           biomassa × ANPP × carbono por marcador
  f5_grassland_compare_menu.png       total e campestre por produto (barra)

Todas as agregações (classe/marcador/total) são derivadas EM MEMÓRIA das
linhas de combinação (classe × def_sec; e meses p/ ANPP) do consolidado —
o CSV do GEE não traz linhas agregadas.
"""
import numpy as np
import pandas as pd  # noqa: F401

print("=" * 68)
print("statistics · 13_figures · figuras")
print("=" * 68)

forest = pd.read_csv(out_dir() / "consolidado_forest_2023.csv", encoding="utf-8")
grassland_raw = pd.read_csv(out_dir() / "consolidado_grassland_2023.csv",
                            encoding="utf-8") if (out_dir() /
                            "consolidado_grassland_2023.csv").exists() else None
ref = CONF["GRASSLAND_REF_VERSION"]
grassland = grassland_raw
if grassland is not None and "versao" in grassland.columns:
    grassland = grassland[grassland["versao"].astype(str) == str(ref)]
    print(f"[INFO] grassland consolidado filtrado p/ referência {ref}")

VEC = CONF["FOREST_VERSIONS"]
X = np.arange(len(VEC))

# --- f1: estoque por classe decimal (n0), agrupado por versão -----------------
if len(forest):
    order = (forest.groupby("classe")["estoque_t"].sum()).sort_values().index
    cls = [n0_label(c) for c in order]
    fig, ax = plt.subplots(figsize=(12, max(4, 0.42 * len(cls))))
    for i, v in enumerate(VEC):
        fs = por_classe(forest[forest["version"] == v])
        vals = fs.groupby("classe")["estoque_t"].sum()
        ax.barh(cls, [vals.get(c, 0) for c in order], label=v,
                color=plt.cm.viridis(i / max(len(VEC) - 1, 1)),
                height=0.8 / len(VEC) - 0.04 if len(VEC) > 1 else 0.8)
    ax.set(xlabel=VALUE_LABEL["estoque_t"],
           title=f"f1 · {VALUE_LABEL['estoque_t']} por classe C11 (2023)")
    ax.grid(axis="x", alpha=0.3)
    ax.legend(ncols=len(VEC))
    save_fig(fig, "f1_forest_estoque_por_classe.png")

    # --- f2: marcador florestal/campestre/outros por versão --------------------
    piv = (pivot_marcador(forest, "estoque_t")
           .pivot(index="version", columns="marcador", values="estoque_t")
           .fillna(0))
    cols = ["florestal", "campestre", "outros"]
    cols = [c for c in cols if c in piv.columns]
    fig, ax = plt.subplots(figsize=(9, 4.5))
    bottom = np.zeros(len(piv))
    for c in cols:
        ax.bar(piv.index, piv[c], bottom=bottom, label=c)
        bottom += piv[c].values
    ax.set(xlabel="versão", ylabel="estoque (t)",
           title="f2 · estoque por marcador (florestal × campestre × outros)")
    ax.legend()
    save_fig(fig, "f2_forest_estoque_por_marcador.png")

    # --- f3: comparativo entre versões (marcador florestal e total) ------------
    cmpf = build_compare_forest(forest)
    cmpf = cmpf[cmpf["marcador"].isin(["florestal", "total"])]
    pivc = cmpf.pivot(index="version", columns="marcador", values="estoque_t")
    fig, ax = plt.subplots(figsize=(8, 4.5))
    pivc.plot(ax=ax, marker="o", color={c: CONF[k] for c, k in
             [("florestal", "C_FOREST"), ("total", "C_RED")]})
    ax.set(xlabel="versão", ylabel="estoque (t)",
           title="f3 · comparativo florestal(101) e total por versão (2023)")
    save_fig(fig, "f3_forest_compare_versoes.png")

# --- f4/f5: grassland ---------------------------------------------------------
if grassland is not None and len(grassland):
    produtos = [p for p in grassland["produto"].unique()]
    fig, axes = plt.subplots(2, 2, figsize=(11, 8))
    axes = axes.ravel()
    for i, p in enumerate(produtos):
        g = grassland[grassland["produto"] == p]
        vc = grass_value_col(p)
        if vc == "anpp_t" and "periodo" in g.columns:
            deagg = (g.groupby(["classe", "marcador"], dropna=False)
                     .agg(anpp_t=("anpp_t", "sum")).reset_index())
        else:
            deagg = g
        marc = deagg.groupby("marcador", dropna=False)[vc].sum()
        tot = deagg[vc].sum()
        ax = axes[i]
        ax.bar(marc.index, marc.values, color=[CONF["C_GRASS"], CONF["C_OTHER"]])
        ax.axhline(tot, color=CONF["C_RED"], ls="--", lw=1)
        ax.set(title=f"{p} · total {tot:.0f} {GRASS_PRODUCT_LABEL.get(p, '')}",
               ylabel="t")
        if len(marc) == 0:
            ax.text(0.5, 0.5, "sem dados", ha="center", va="center", alpha=0.5)
    fig.suptitle(f"f4 · produtos campestres {ref} por marcador (2023)",
                 fontweight="bold")
    for ax in axes[len(produtos):]:
        ax.axis("off")
    save_fig(fig, "f4_grassland_produtos.png")

    cmpg = build_compare_grassland(grassland)
    totp = cmpg[cmpg["n0"] == "TOTAL"]
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.bar(totp["produto"], totp["valor_t"], color=CONF["C_GRASS"])
    ax.set(xlabel="produto", ylabel="t",
           title=f"f5 · grassland {ref} — total por produto (2023)")
    save_fig(fig, "f5_grassland_compare_menu.png")

# ---------------------------------------------------------------------------
# Densidade ponderada (sum(t)/sum(ha)) + medidas das REFERÊNCIAS
# ---------------------------------------------------------------------------
try:
    refs = load_references_csv()
except Exception as e:  # noqa: BLE001
    print("[AVISO] referências indisponíveis:", e)
    refs = pd.DataFrame()


def _refs_por_classe(term):
    """Densidade de referência (na unidade do produto) por classe × ref."""
    if not len(refs):
        return pd.DataFrame(columns=["classe", "ref_id", "dens"])
    allowed = ALLOWED_TERMS.get(term, ())
    sub = refs[refs["ref_term"].isin(allowed)].copy()
    if not len(sub):
        return pd.DataFrame(columns=["classe", "ref_id", "dens"])
    sub = ref_value_for_product(sub, term).dropna(subset=["ref_value"])
    return sub[["classe", "ref_id", "ref_value"]].rename(
        columns={"ref_value": "dens"})


def _plot_dens_refs(ax, refsub, order):
    """Sobrepõe as densidades de referência (pontos) num eixo de barras."""
    for rid, g in refsub.groupby("ref_id"):
        gi = g.set_index("classe")
        ys = [gi["dens"].get(c, np.nan) for c in order]
        ax.plot(range(len(order)), ys, marker="o", ls="none", ms=5,
                label=f"ref: {rid}")


def _density_figure(ax_order_by_area, value_col, keys, ylab, title, fname,
                    ref_term=None):
    d = dens_ponderada(ax_order_by_area, value_col, keys)
    if not len(d):
        return
    order = (ax_order_by_area.groupby("classe")["area_ha"].sum()
             .sort_values(ascending=False).index[:15])
    cls = [n0_label(c) for c in order]
    x = np.arange(len(order))
    fig, ax = plt.subplots(figsize=(12, max(4, 0.45 * len(cls))))
    if "version" in keys:
        for i, v in enumerate(CONF["FOREST_VERSIONS"]):
            dv = d[d["version"] == v].set_index("classe")["dens"]
            ax.bar(x + i * 0.8 / len(CONF["FOREST_VERSIONS"]),
                   [dv.get(c, np.nan) for c in order],
                   width=0.8 / len(CONF["FOREST_VERSIONS"]), label=v)
        ax.set_xticks(x + 0.4 - 0.4 / len(CONF["FOREST_VERSIONS"]))
    else:
        dv = d.set_index("classe")["dens"]
        ax.bar(x, [dv.get(c, np.nan) for c in order], label=keys[-1])
        ax.set_xticks(x)
    if ref_term:
        _plot_dens_refs(ax, _refs_por_classe(ref_term), order)
    ax.set_xticklabels(cls, rotation=45, ha="right")
    ax.set(ylabel=ylab, title=title)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(ncols=2, fontsize=8)
    save_fig(fig, fname)


if len(forest):
    _density_figure(forest, "estoque_t", ["version", "classe"],
                    VALUE_DENS_LABEL["estoque_t"],
                    f"f6 · densidade ponderada de {VALUE_LABEL['estoque_t']} "
                    "por classe × versão + referências",
                    "f6_forest_densidade_por_classe.png", ref_term="agc")

if grassland_raw is not None and len(grassland_raw):
    _vers = (sorted(grassland_raw["versao"].astype(str).unique())
             if "versao" in grassland_raw.columns else [])
    g_src = grassland_raw
    if str(ref) in _vers:
        g_src = grassland_raw[grassland_raw["versao"].astype(str) == str(ref)]
    elif _vers:
        g_src = grassland_raw[grassland_raw["versao"].astype(str) == _vers[0]]
        print(f"[INFO] grassland densidade usando versão {_vers[0]}")
    gC = g_src[g_src["produto"] == "carbon"]
    gB = g_src[g_src["produto"] == "biomass"]
    if len(gC):
        _density_figure(gC, "estoque_t", ["classe"],
                        VALUE_DENS_LABEL["estoque_t"],
                        f"f7 · densidade ponderada de "
                        f"{VALUE_LABEL['estoque_t']} — campestre + referências",
                        "f7_grassland_carbono_densidade.png", ref_term="carbon")
    if len(gB):
        _density_figure(gB, "biomassa_t", ["classe"],
                        VALUE_DENS_LABEL["biomassa_t"],
                        f"f8 · densidade ponderada de "
                        f"{VALUE_LABEL['biomassa_t']} — campestre + referências",
                        "f8_grassland_biomassa_densidade.png", ref_term="biomass")

# ---------------------------------------------------------------------------
# Grids (heatmaps) com os valores dentro: total (t), área (ha) e t/ha
# ---------------------------------------------------------------------------
def _top_classes(df, n=12):
    if df is None or not len(df) or "n0" not in df.columns:
        return []
    return (df.groupby("n0")["area_ha"].sum().sort_values(ascending=False)
            .index[:n].tolist())


def _grid_panel(ax, piv, title, fmt, cmap):
    if piv is None or not len(piv):
        ax.axis("off")
        ax.set_title(title, fontsize=10)
        return
    mat = piv.values.astype(float)
    im = ax.imshow(mat, aspect="auto", cmap=cmap)
    ax.set_xticks(range(piv.shape[1]))
    ax.set_xticklabels([str(c) for c in piv.columns], rotation=45, ha="right",
                       fontsize=8)
    ax.set_yticks(range(piv.shape[0]))
    ax.set_yticklabels([str(r) for r in piv.index], fontsize=8)
    for i in range(mat.shape[0]):
        for j in range(mat.shape[1]):
            v = mat[i, j]
            if np.isfinite(v):
                ax.text(j, i, format(v, fmt), ha="center", va="center",
                        fontsize=7)
    ax.set_title(title, fontsize=10)
    plt.colorbar(im, ax=ax, shrink=0.8)


def _grid_pivots(rows, col_col, val_col):
    t = rows.pivot_table(index="n0", columns=col_col, values=val_col,
                         aggfunc="sum", dropna=False)
    a = rows.pivot_table(index="n0", columns=col_col, values="area_ha",
                         aggfunc="sum", dropna=False)
    d = t / a.replace(0, np.nan)
    return t, a, d


f_t = f_a = f_d = None
if len(forest):
    fr = forest[forest["n0"].isin(_top_classes(forest, 12))]
    f_t, f_a, f_d = _grid_pivots(fr, "version", "estoque_t")

g_t = g_a = g_d = None
if grassland_raw is not None and len(grassland_raw):
    gr = grassland_raw.copy()
    _vcols = [c for c in ("estoque_t", "biomassa_t", "anpp_t")
              if c in gr.columns]
    gr["valor"] = gr[_vcols].sum(axis=1, min_count=1)
    gr = gr[gr["n0"].isin(_top_classes(gr, 12))]
    g_t, g_a, g_d = _grid_pivots(gr, "produto", "valor")

for piv_f, piv_g, fmt, cmap, fname, lab in [
    (f_t, g_t, ",.0f", "YlGnBu", "g1_grid_total.png", "total (t)"),
    (f_a, g_a, ",.0f", "YlOrBr", "g2_grid_area.png", "área (ha)"),
    (f_d, g_d, ",.1f", "YlOrRd", "g3_grid_densidade.png", "densidade (t/ha)"),
]:
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    _grid_panel(axes[0], piv_f, f"forest · {lab} — classe × versão", fmt, cmap)
    _grid_panel(axes[1], piv_g, f"grassland · {lab} — classe × produto", fmt,
                cmap)
    fig.suptitle(f"grid · {lab}", fontweight="bold")
    save_fig(fig, fname)

print()
print(f"[ARTEFATOS] {len(manifest())} — baixe/exporte as figuras para cá:")
for m in manifest():
    print("  ", m)
print()
print("[DICA] figuras → COLLECTION_00/toolkits/statistics/v00/run/cells_for_collab_notebook/figures/ no repo")
print("=" * 68)

In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 14_report.py
===================================================================
Etapa 14 · Relatório consolidado (XLSX + .md). Rode após 13_figures.

Gera, em OUT_DIR:
  resumo_pantanal_2023.xlsx — abas:
    forest: n0 por versão | n1 | marcador | total | comparativo_versoes
    grassland: n0 por produto | n1 | marcador | total | comparativo_produtos
  resumo_pantanal_2023.md — sumário textual (totais, marcador, QA)

Todas as abas agregadas (n1/marcador/total) são derivadas EM MEMÓRIA das
linhas de combinação (classe × def_sec; e meses p/ NPP) do consolidado.
"""
import pandas as pd

print("=" * 68)
print("statistics · 14_report · relatório")
print("=" * 68)

forest = pd.read_csv(out_dir() / "consolidado_forest_2023.csv", encoding="utf-8")
ref = CONF["GRASSLAND_REF_VERSION"]
grassland_path = out_dir() / "consolidado_grassland_2023.csv"
grassland = pd.read_csv(grassland_path, encoding="utf-8") if grassland_path.exists() else None
if grassland is not None and "versao" in grassland.columns:
    grassland = grassland[grassland["versao"].astype(str) == str(ref)]
cmpf = build_compare_forest(forest)
cmpg = build_compare_grassland(grassland)

md = [f"# Resumo das estatísticas · Pantanal · ano {CONF['YEAR']}",
      f"\nGerado por `toolkits/statistics/v00/run/cells_for_collab_notebook` em Colab.",
      f"\nFiltro def_sec: "
      f"{', '.join(CONF['DEF_SEC_FILTER']) if CONF.get('DEF_SEC_FILTER') else 'todos'}.",
      f"\n## Floresta ({VALUE_LABEL['estoque_t']})\n"]

xlsx_path = out_dir() / f"resumo_pantanal_{CONF['YEAR']}.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
    # ---------------- FLORESTA ----------------
    if len(forest) and "version" in forest.columns:
        has_inc = "incerteza_t" in forest.columns
        for v in CONF["FOREST_VERSIONS"]:
            d = forest[forest["version"] == v]
            if not len(d):
                continue
            agg = {"area_ha": ("area_ha", "sum"),
                   "estoque_t": ("estoque_t", "sum")}
            if has_inc:
                agg["incerteza_t"] = ("incerteza_t", "sum")
            tab = (d.groupby(["territorio", "classe", "n0"])
                   .agg(**agg)
                   .reset_index())
            tab.to_excel(xw, sheet_name=f"{v}_n0", index=False)
            (d.groupby(["territorio", "n1"])
             .agg(**agg)
             .reset_index().to_excel(xw, sheet_name=f"{v}_n1", index=False))
            (d.groupby(["territorio", "marcador"])
             .agg(**agg)
             .reset_index().to_excel(xw, sheet_name=f"{v}_marcador",
                                     index=False))
            (d.groupby(["territorio"])
             .agg(**agg)
             .reset_index().to_excel(xw, sheet_name=f"{v}_total", index=False))
            tot = d["estoque_t"].sum()
            fl = forest[(forest["version"] == v) &
                        (forest["marcador"] == "florestal")]["estoque_t"].sum()
            md.append(f"- **{v}**: total {tot:,.0f} t · florestal"
                      f" {fl:,.0f} t ({fl / tot * 100:.1f}%)")
        if cmpf is not None and len(cmpf):
            cmpf.to_excel(xw, sheet_name="comparativo_versoes", index=False)
            md.append("\n### Comparativo florestal (101) e total por versão\n")
            for _, r in cmpf[cmpf["marcador"] == "florestal"].iterrows():
                md.append(f"- {r['version']}: {r['estoque_t']:,.0f} t"
                          f" (área {r['area_ha']:,.0f} ha)")
    # ---------------- GRASSLAND ----------------
    if grassland is not None and len(grassland):
        md.append(f"\n## Campestre ({ref} — biomassa/NPP/carbono, "
                  f"{CONF['YEAR']})\n")
        for p in grassland["produto"].unique():
            d = grassland[grassland["produto"] == p]
            vc = grass_value_col(p)
            if vc == "anpp_t" and "periodo" in d.columns:
                d = (d.groupby(["territorio", "classe", "n0", "n1", "n2",
                                "marcador"], dropna=False)
                     .agg(area_ha=("area_ha", "sum"),
                          anpp_t=("anpp_t", "sum")).reset_index())
            rows = d.rename(columns={vc: "valor_t"})
            tot = rows["valor_t"].sum()
            camp = rows[rows["marcador"] == "campestre"]["valor_t"].sum()
            d0 = (rows.groupby(["territorio", "classe", "n0"])
                  .agg(area_ha=("area_ha", "sum"), valor_t=("valor_t", "sum"))
                  .reset_index())
            d0.to_excel(xw, sheet_name=f"{p}_n0", index=False)
            (rows.groupby(["territorio", "n1"])
             .agg(area_ha=("area_ha", "sum"), valor_t=("valor_t", "sum"))
             .reset_index().to_excel(xw, sheet_name=f"{p}_n1", index=False))
            (rows.groupby(["territorio", "marcador"])
             .agg(area_ha=("area_ha", "sum"), valor_t=("valor_t", "sum"))
             .reset_index().to_excel(xw, sheet_name=f"{p}_marcador",
                                     index=False))
            (rows.groupby(["territorio"])
             .agg(area_ha=("area_ha", "sum"), valor_t=("valor_t", "sum"))
             .reset_index().to_excel(xw, sheet_name=f"{p}_total", index=False))
            md.append(f"- **{p}**: {tot:,.0f} "
                      f"{GRASS_PRODUCT_LABEL.get(p, 't')}"
                      f" · campestre {camp:,.0f} t"
                      f" ({camp / tot * 100:.1f}%)")
        if cmpg is not None and len(cmpg):
            cmpg.to_excel(xw, sheet_name="comparativo_produtos", index=False)

report_md = out_dir() / f"resumo_pantanal_{CONF['YEAR']}.md"
write_artifact(report_md, "\n".join(md))
print()
print("[RESUMO]")
print("\n".join(md))
print()
print(f"[ARTEFATOS] {len(manifest())} — " + ", ".join(manifest()[-2:]))
print("=" * 68)

In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 15_references.py
====================================================================
Etapa 15 · Análise das REFERÊNCIAS (cruzes T1/T2 por termo). Rode após
11_consolidate (precisa dos consolidados) e 06_references (catálogo/mapa).

Lê os CSVs AGRUPADOS (`stats_references_<grupo>_2023.csv`) via
load_references_csv() e produz, em OUT_DIR:
  1. referencias_por_classe_2023.csv (cache) — refs longas por classe
  2. tabela_cross_2023.csv — produto×versão×classe×referência (T1)
  3. tabela_melhor_ano_referencia_2023.csv — ref+época de menor distância (T2)
  4. resumo_referencia_2023.md — achados e flags (AGB→C, cobertura campestre)
"""
print("=" * 68)
print("statistics · 15_references · análise de dados de referência")
print("=" * 68)

# Confirma constantes via CONF (fonte única de verdade operacional — a cell00
# já derivou CARBON_FRACTION e REF_MIN_VALID de CONF).
assert CARBON_FRACTION == CONF["CARBON_FRACTION"]
assert REF_MIN_VALID == CONF["REF_MIN_VALID"]

# --- entrada (consolidados da cell02) ----------------------------------------
forest = pd.read_csv(out_dir() / "consolidado_forest_2023.csv", encoding="utf-8")
grass_path = out_dir() / "consolidado_grassland_2023.csv"
for p in (out_dir() / "consolidado_forest_2023.csv", grass_path):
    if not p.exists():
        print(f"[FALTA] {p} — rode a cell02 antes (gera os consolidados).")
if not grass_path.exists():
    print("[INFO] sem grassland — cruzamento só forest.")
grass = pd.read_csv(grass_path, encoding="utf-8") if grass_path.exists() else None

print(f"[IN] forest {len(forest)} linhas · grassland "
      f"{len(grass) if grass is not None else 0} linhas")

# --- 1) referências por classe (lê CSVs do GEE + cache em OUT_DIR) ----------
cache = out_dir() / "referencias_por_classe_2023.csv"
if cache.exists():
    refs_df = pd.read_csv(cache)
    print(f"[CACHE] {cache} ({len(refs_df)} linhas) — apague o cache para "
          "reler os CSVs do Drive")
else:
    print(f"[CSV] lendo stats_references_*_{CONF['YEAR']}.csv de"
          f" {csv_dir()} · território: {CONF['REF_TERRITORIO']}")
    refs_df = load_references_csv(lulc_year=CONF["YEAR"])
    refs_df.to_csv(cache, index=False)
    print(f"[OUT] {cache} ({len(refs_df)} linhas)")

# --- QA das referências ------------------------------------------------------
problems = qa_references(refs_df)
for p in problems:
    print("[QA !]", p)
if not problems:
    print("[QA OK] referências: todas com área > 0 nas classes cobertas")

# --- linhas dos produtos (densidade por classe, 2023) ------------------------
products = product_lines(forest, grass, year=CONF["YEAR"])
print(f"[PROD] {len(products)} linhas produto×versão×classe · "
      f"produtos: {sorted(products['produto'].unique())}")

# --- 2) T1 · cross 2023 ------------------------------------------------------
t1 = cross_2023(products, refs_df)
t1_path = out_dir() / "tabela_cross_2023.csv"
t1.to_csv(t1_path, index=False)
print(f"[OUT] {t1_path} ({len(t1)} linhas)")

# --- 3) T2 · melhor ano de referência ----------------------------------------
t2 = best_reference_year(products, refs_df,
                         min_valid=CONF["REF_MIN_VALID"])
t2_path = out_dir() / "tabela_melhor_ano_referencia_2023.csv"
t2.to_csv(t2_path, index=False)
print(f"[OUT] {t2_path} ({len(t2)} linhas)")

# --- 4) resumo markdown ------------------------------------------------------
lines = []
lines.append("# MAPBIOMAS-CARBONO Pantanal — análise de dados de referência (2023)")
lines.append("")
lines.append("Termos comuns: densidade **t C/ha** por classe; biomassa (t MS/ha)")
lines.append("comparada com AGB (lenhoso). Conversão AGB→C com fator %.2f."
             % CARBON_FRACTION)
lines.append("")
lines.append("## T1 · volumes por marcador × grupo de referência")
if len(t1):
    g1 = t1.groupby(["marcador", "ref_group"])["diff_pct"].describe()
    lines.append(g1.to_string())
    lines.append("")
    dens = t1.groupby("produto")["dens_prod"].describe()
    lines.append("## Produto — densidade por classe")
    lines.append(dens.to_string())
    lines.append("")
if len(t2):
    prov = t1[t1["produto"] == "forest"]
    if len(prov):
        fi = prov.groupby("ref_id")["diff_pct"].agg(
            ["median"]).sort_values("median")
        lines.append("## Viés do produto florestal (agbd tratado como C)")
        lines.append("diff_pct mediana por referência (|produto - ref|/prod):")
        lines.append(fi.to_string())
        lines.append("")
    lines.append("## T2 · distribuição do melhor ano de referência")
    if t2["melhor_ref"].notna().any():
        g2 = (t2.dropna(subset=["melhor_ref"])
               .groupby(["produto", "melhor_ref"])["melhor_year"]
               .agg(lambda s: s.value_counts().to_dict()))
        lines.append(g2.to_string())
        lines.append("")
        imp = (t2.dropna(subset=["fator_impl"])
               .groupby("produto")["fator_impl"].agg(["median", "min", "max"]))
        lines.append("## fator_impl (dens_prod / dens_ref — diagnóstico de unidade)")
        lines.append(imp.to_string())
lines.append("")
lines.append("## Flags")
lines.append("- Xu2021 `b1..b20` -> anos 2000..2019 é PREMISSA (metadado ausente).")
lines.append("- AGB refs cobrem vegetação LENHOSA; campestre pode ficar sem")
lines.append("  cobertura (QA `ref_val_pct`).")
lines.append("- Produto florestal `fator_impl` mediano ~2 → confere C vs AGB.")
md = "\n".join(lines) + "\n"
md_path = out_dir() / "resumo_referencia_2023.md"
write_artifact(md_path, md)

print(f"[ARTEFATOS] {len(manifest())} gerados")
print("=" * 68)

In [ ]:
"""
MAPBIOMAS-CARBONO · toolkit statistics/v00 — 16_maps.py
================================================================
Etapa 16 (OPCIONAL) · Mapas espaciais das estatísticas. Rode após 14_report —
somente se CONF["FAZER_MAPAS"] for True.

Diferente das demais etapas, ESTA consulta o Earth Engine: calcula a MÉDIA da
predição florestal (versão `CONF["MAP_FOREST_VERSION"]`) e a FRAÇÃO campestre
(C11: 11/12/50) por grade de GRID_M (default 10 km), recortadas no bioma, e
salva dois PNG em FIG_DIR. Requer EE autenticado e os assets existentes.

Configurável por CONF: `MAP_FOREST_VERSION` (ex.: 'v07'), `MAP_ASSET`
(override do asset) e `MAP_BAND` (override da banda). Para v08+ o prefixo do
asset é `agb_pred_*` (ver docs/BAND_NAMING.md); v03–v07 usam `agc_pred_*`.
"""
print("=" * 68)
print("statistics · 16_maps · mapas (opcional)")
print("=" * 68)

if not CONF["FAZER_MAPAS"]:
    print("[OFF] FAZER_MAPAS=False (padrão). Nada a fazer.")
    print("      Para ativar: CONF['FAZER_MAPAS'] = True acima desta célula")
    print("      e rode novamente (custa uma redução no EE).")
    print("=" * 68)
    raise SystemExit(0)

import io  # noqa: E402

import ee  # noqa: E402
import requests  # noqa: E402
from PIL import Image  # noqa: E402

BIOME = ee.FeatureCollection(
    "projects/ee-ipam/assets/ORIGINAIS/IBGE/limite_biomas_IBGE_2026")\
    .filter(ee.Filter.eq("Bioma", "Pantanal")).first().geometry()

_FV = CONF.get("MAP_FOREST_VERSION", "v07")
_prefix = "agb_pred" if int(_FV.lstrip("v")) >= 8 else "agc_pred"
ASSET = CONF.get("MAP_ASSET") or (
    "projects/mapbiomas-brazil/assets/CARBON/MAPPING/COLLECTION_00/"
    f"PANTANAL_FOREST/{_FV.upper()}/PREDICTIONS/"
    f"{_prefix}_{_FV}_merged_{CONF['YEAR']}")
BAND = CONF.get("MAP_BAND", f"prediction_{CONF['YEAR']}")
print(f"[EE] asset: {ASSET} · banda: {BAND}")

MOSAIC = ee.Image(ASSET).select(BAND)
LULC = ee.Image(
    "projects/mapbiomas-public/assets/brazil/lulc/collection11/"
    "mapbiomas_brazil_collection11_coverage_v3").select(
    f"classification_{CONF['YEAR']}")


def thumb(img, params, name):
    url = img.getThumbURL(params)
    pil = Image.open(io.BytesIO(requests.get(url).content))
    out = fig_dir() / name
    pil.save(out)
    print(f"[FIG] {out}")


g = CONF["GRID_M"]
agc = MOSAIC.reduceResolution(ee.Reducer.mean(), maxPixels=1024)\
    .reproject("EPSG:4326", None, g).clip(BIOME)
camp = LULC.remap([11, 12, 50], [1], 0)\
    .reduceResolution(ee.Reducer.mean(), maxPixels=1024)\
    .reproject("EPSG:4326", None, g).clip(BIOME)

thumb(agc, {"min": 0, "max": 350, "palette": ["#0b4463", "#1a9850", "#fee08b"]},
      f"mapa_agb_{_FV}_grid_{g // 1000}km.png")
thumb(camp, {"min": 0, "max": 1, "palette": ["#ffffff", "#2b83ba"]},
      f"mapa_campestre_grid_{g // 1000}km.png")
print(f"[OK] grid de {g // 1000} km — mapas salvos em {fig_dir()}")
print("=" * 68)
